#An AI-Powered Video Intelligence System

Built with LangChain + RAG + LLMs

In [3]:
!pip install -q youtube-transcript-api \
    langchain \
    langchain-community \
    langchain-huggingface \
    huggingface_hub \
    faiss-cpu \
    python-dotenv \
    numpy


In [5]:
import os
import numpy as np

from dotenv import load_dotenv

from youtube_transcript_api import (
    YouTubeTranscriptApi,
    TranscriptsDisabled
)

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS

from langchain_core.prompts import PromptTemplate

from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda
)

from langchain_core.output_parsers import StrOutputParser

from huggingface_hub import InferenceClient

from langchain_huggingface import (
    HuggingFaceEndpoint,
    ChatHuggingFace
)


/tmp/ipykernel_3104/3231121038.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [19]:
os.environ["HF_TOKEN"] = ""

In [20]:
load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN is not set. Please add your Hugging Face API token."
    )

print("Hugging Face token loaded successfully.")

Hugging Face token loaded successfully.


 Fetch YouTube Transcript

In [59]:
video_id = "W4VmJ8UaUjE"

try:
    fetched_transcript = YouTubeTranscriptApi().fetch(video_id, languages=['hi'])
    transcript_list = fetched_transcript.to_raw_data()
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")
except Exception as e:
    print(f"An error occurred: {e}")

मेरे पॉडकास्ट में जितने भी गेस्ट आए सबने बोला है कि आईपीओ में पैसा बनता ही नहीं। यार मेरी फैमिली हैज़ बीन इन्वेस्टिंग इन आईपीओ मार्केट फ्रॉम 1980। मेरे दादाजी ने अपनी बेटी की शादी सिर्फ आईपीओ इन्वेस्टिंग से कर करके करवाई है। आज के हमारे गेस्ट है अनंत लट्ठा जिन्हें YouTube पर 3 मिलियन से ज्यादा लोग फॉलो करते हैं। [संगीत] आज हम डिस्कस करने वाले हैं कि आईपीओ में अलॉटमेंट कैसे मिल सकता है? आईपीओ में कैसे पैसे बना सकते हैं? एंड मोस्टेंटली आईपीओ में इन्वेस्ट करने के पहले क्या चीजें देखनी चाहिए? काफी सारे ये लोग सोचते हैं कि भैया अगर मेरे को आईपीओ में अलॉटमेंट चाहिए तो मैं मेरे नाम से एक लॉट ही अप्लाई कर दूंगा। मम्मी के नाम से दो से, पापा के नाम से तीन से, चाचा के नाम से चार से। और उनका यह इल्लुजन होता है कि मार्केट में क्या करती है? जो कंपनी जब लॉटरी बेसिस पे अलॉटमेंट देती है वो सबसे पहले एक लॉट वालों को अलग करती है, दो लॉट वालों को अलग करती है, तीन को अलग करती है, पांच को अलग करती है। फिर उसमें से लॉटरी निकालती है। ऐसा होता ही नहीं है। आजकल फंडामेंटल एनालिसिस करता कौन है? सब जीएनपी देखते हैं

 Text Splitting

In [60]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.create_documents(
    [transcript]
)

print()
print("Number of chunks:", len(chunks))

print()
print("Example chunk:")
print(chunks[0].page_content)


Number of chunks: 85

Example chunk:
मेरे पॉडकास्ट में जितने भी गेस्ट आए सबने बोला है कि आईपीओ में पैसा बनता ही नहीं। यार मेरी फैमिली हैज़ बीन इन्वेस्टिंग इन आईपीओ मार्केट फ्रॉम 1980। मेरे दादाजी ने अपनी बेटी की शादी सिर्फ आईपीओ इन्वेस्टिंग से कर करके करवाई है। आज के हमारे गेस्ट है अनंत लट्ठा जिन्हें YouTube पर 3 मिलियन से ज्यादा लोग फॉलो करते हैं। [संगीत] आज हम डिस्कस करने वाले हैं कि आईपीओ में अलॉटमेंट कैसे मिल सकता है? आईपीओ में कैसे पैसे बना सकते हैं? एंड मोस्टेंटली आईपीओ में इन्वेस्ट करने के पहले क्या चीजें देखनी चाहिए? काफी सारे ये लोग सोचते हैं कि भैया अगर मेरे को आईपीओ में अलॉटमेंट चाहिए तो मैं मेरे नाम से एक लॉट ही अप्लाई कर दूंगा। मम्मी के नाम से दो से, पापा के नाम से तीन से, चाचा के नाम से चार से। और उनका यह इल्लुजन होता है कि मार्केट में क्या करती है? जो कंपनी जब लॉटरी बेसिस पे अलॉटमेंट देती है वो सबसे पहले एक लॉट वालों को अलग करती है, दो लॉट वालों को अलग करती है, तीन को अलग करती है, पांच को अलग करती है। फिर उसमें से लॉटरी निकालती है। ऐसा होता ही नहीं है। आजकल फंडामेंटल एना

**Hugging Face Embeddings**

We are using Hugging Face's API.
The embedding model:
BAAI/bge-small-en-v1.5
This converts text into numerical vectors.
We are NOT downloading the model locally.

In [62]:
class HuggingFaceAPIEmbeddings:

    def __init__(
        self,
        model="BAAI/bge-small-en-v1.5",
        api_key=None,
        provider="hf-inference"
    ):
        self.model = model
        self.client = InferenceClient(
            provider=provider,
            api_key=api_key
        )

    def embed_documents(self, texts):
        embeddings = []

        for text in texts:
            result = self.client.feature_extraction(
                text,
                model=self.model
            )

            result = np.asarray(result, dtype=np.float32)

            if result.ndim == 2:
                result = result.mean(axis=0)

            # Normalize
            norm = np.linalg.norm(result)

            if norm != 0:
                result = result / norm

            embeddings.append(result.tolist())

        return embeddings

    def embed_query(self, text):
        result = self.client.feature_extraction(
            text,
            model=self.model
        )

        result = np.asarray(result, dtype=np.float32)

        if result.ndim == 2:
            result = result.mean(axis=0)

        # Normalize
        norm = np.linalg.norm(result)

        if norm != 0:
            result = result / norm

        return result.tolist()

    # IMPORTANT
    def __call__(self, text):
        return self.embed_query(text)


In [65]:
embeddings = HuggingFaceAPIEmbeddings(
    model="BAAI/bge-small-en-v1.5",
    api_key=HF_TOKEN,
    provider="hf-inference"
)

Create FAISS Vector Store

In [64]:
vector_store = FAISS.from_documents(
    chunks,
    embeddings
)

print()
print("FAISS vector store created successfully.")



FAISS vector store created successfully.


Create Retriever

In [66]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

In [67]:
retriever.invoke("What is ipo?")

[Document(id='1066094d-bf68-4548-83b6-f9a4a2713cdc', metadata={}, page_content='विल अंडर परफॉर्म। रादर मैं तो 2030 तक बोल सकता हूं कि यूएस इज़ सेट टू अंडर परफॉर्म। यूएस हैज़ बीन दी वर्स्ट परफॉर्म बेस्ट परफॉर्मिंग मार्केट। इट इज़ पार्ट ऑफ़ द गेम। यूएस विल अंडर परफॉर्म इवेंचुअली। और इसकी अंडर परफॉर्मेंस का कारण हो सकता है कि एआई बबल का बस्ट हो सकता है 6 आठ महीने नहीं करे। बट आप रिलेटिव साल में 3 साल में देखोगे आपको लगेगा यूएस विल अंडर परफॉर्म। सेकंड इंडिया विल परफॉर्म बेटर देन अदर इमर्जिंग मार्केट्स। बेटर का मतलब क्या होता है? क्वार्टाइल वन में आपकी परफॉर्मेंस आ जाएगी जो अभी क्वार्टल फोर में चल रही है। इंडिया विल कम इन क्वटाइल वन परफॉर्मेंस। तीसरा मुझे लिखने में थोड़ा सा रिस्की लग रहा है और यह लास्ट है। अभी जैसे आईएआर डेप्रिशिएट हो रहा है ना हम यह है अगले एक साल के लिए वन ईयर का आई एम एक्सपेक्टिंग कि आईएआर सपोर्टेड एसेट्स विल परफॉर्म बेटर देन एनी अदर एसेट्स एंड आईएआर विल अप्रिशिएट इज अ डिफिकल्ट वर्ड टू से बट एटलीस्ट स्टेबलाइज जितना लोग बात कर रहे हैं ना $120 पे पहुंच जाएगा 130 बार भाई नहीं प

Hugging Face LLM

In [68]:
llm_endpoint = HuggingFaceEndpoint(

    repo_id="Qwen/Qwen2.5-Coder-3B-Instruct",

    task="text-generation",

    huggingfacehub_api_token=HF_TOKEN,

    max_new_tokens=512,

    temperature=0.2
)


Convert endpoint into Chat model

In [69]:
llm = ChatHuggingFace(
    llm=llm_endpoint
)

print()
print("Hugging Face LLM initialized.")


Hugging Face LLM initialized.


Prompt Template

In [70]:
prompt = PromptTemplate(

    template="""

You are a helpful assistant answering questions
about a YouTube video.

Answer ONLY using the provided transcript context.

Rules:

1. Do not use outside knowledge.

2. If the answer is not present in the context,
   say:

   "I don't know based on the provided transcript."

3. Give a clear and concise answer.

4. If the question asks whether a topic was discussed,
   clearly say whether it was discussed.

Transcript Context:

{context}


Question:

{question}


Answer:

""",

    input_variables=[
        "context",
        "question"
    ]
)


Manual RAG Pipeline

In [71]:
question = """
Is the topic of ipo discussed in this video?
If yes, what was discussed?
"""


# Retrieve relevant chunks

retrieved_docs = retriever.invoke(
    question
)


# Combine retrieved chunks

context_text = "\n\n".join(

    doc.page_content

    for doc in retrieved_docs

)


print()
print("Retrieved Context:")
print(context_text)



Retrieved Context:
विल अंडर परफॉर्म। रादर मैं तो 2030 तक बोल सकता हूं कि यूएस इज़ सेट टू अंडर परफॉर्म। यूएस हैज़ बीन दी वर्स्ट परफॉर्म बेस्ट परफॉर्मिंग मार्केट। इट इज़ पार्ट ऑफ़ द गेम। यूएस विल अंडर परफॉर्म इवेंचुअली। और इसकी अंडर परफॉर्मेंस का कारण हो सकता है कि एआई बबल का बस्ट हो सकता है 6 आठ महीने नहीं करे। बट आप रिलेटिव साल में 3 साल में देखोगे आपको लगेगा यूएस विल अंडर परफॉर्म। सेकंड इंडिया विल परफॉर्म बेटर देन अदर इमर्जिंग मार्केट्स। बेटर का मतलब क्या होता है? क्वार्टाइल वन में आपकी परफॉर्मेंस आ जाएगी जो अभी क्वार्टल फोर में चल रही है। इंडिया विल कम इन क्वटाइल वन परफॉर्मेंस। तीसरा मुझे लिखने में थोड़ा सा रिस्की लग रहा है और यह लास्ट है। अभी जैसे आईएआर डेप्रिशिएट हो रहा है ना हम यह है अगले एक साल के लिए वन ईयर का आई एम एक्सपेक्टिंग कि आईएआर सपोर्टेड एसेट्स विल परफॉर्म बेटर देन एनी अदर एसेट्स एंड आईएआर विल अप्रिशिएट इज अ डिफिकल्ट वर्ड टू से बट एटलीस्ट स्टेबलाइज जितना लोग बात कर रहे हैं ना $120 पे पहुंच जाएगा 130 बार भाई नहीं पहुंचेगा यूएस हैज़ परफॉर्म्ड यूएस एंड इट इज़ नथिंग कि मैं कोई ए

Create Final Prompt

In [72]:
final_prompt = prompt.invoke({

    "context": context_text,

    "question": question

})


print()
print("Final Prompt:")
print(final_prompt)



Final Prompt:
text='\n\nYou are a helpful assistant answering questions\nabout a YouTube video.\n\nAnswer ONLY using the provided transcript context.\n\nRules:\n\n1. Do not use outside knowledge.\n\n2. If the answer is not present in the context,\n   say:\n\n   "I don\'t know based on the provided transcript."\n\n3. Give a clear and concise answer.\n\n4. If the question asks whether a topic was discussed,\n   clearly say whether it was discussed.\n\nTranscript Context:\n\nविल अंडर परफॉर्म। रादर मैं तो 2030 तक बोल सकता हूं कि यूएस इज़ सेट टू अंडर परफॉर्म। यूएस हैज़ बीन दी वर्स्ट परफॉर्म बेस्ट परफॉर्मिंग मार्केट। इट इज़ पार्ट ऑफ़ द गेम। यूएस विल अंडर परफॉर्म इवेंचुअली। और इसकी अंडर परफॉर्मेंस का कारण हो सकता है कि एआई बबल का बस्ट हो सकता है 6 आठ महीने नहीं करे। बट आप रिलेटिव साल में 3 साल में देखोगे आपको लगेगा यूएस विल अंडर परफॉर्म। सेकंड इंडिया विल परफॉर्म बेटर देन अदर इमर्जिंग मार्केट्स। बेटर का मतलब क्या होता है? क्वार्टाइल वन में आपकी परफॉर्मेंस आ जाएगी जो अभी क्वार्टल फोर में चल रही है

Generate Answer

In [73]:
answer = llm.invoke(
    final_prompt
)

print()
print("FINAL ANSWER:")
print(answer.content)



FINAL ANSWER:
Yes, the topic of IPO (Initial Public Offering) was discussed in this video.


BUILD COMPLETE LCEL RAG CHAIN

In [74]:
def format_docs(retrieved_docs):

    return "\n\n".join(

        doc.page_content

        for doc in retrieved_docs

    )



Parallel chain

In [75]:
parallel_chain = RunnableParallel({

    "context":
        retriever
        |
        RunnableLambda(format_docs),

    "question":
        RunnablePassthrough()

})


Test Parallel Chain

In [76]:
test_question = "what is ipo?"

result = parallel_chain.invoke(
    test_question
)

print()
print("Context from Parallel Chain:")
print(result["context"])


Context from Parallel Chain:
विल अंडर परफॉर्म। रादर मैं तो 2030 तक बोल सकता हूं कि यूएस इज़ सेट टू अंडर परफॉर्म। यूएस हैज़ बीन दी वर्स्ट परफॉर्म बेस्ट परफॉर्मिंग मार्केट। इट इज़ पार्ट ऑफ़ द गेम। यूएस विल अंडर परफॉर्म इवेंचुअली। और इसकी अंडर परफॉर्मेंस का कारण हो सकता है कि एआई बबल का बस्ट हो सकता है 6 आठ महीने नहीं करे। बट आप रिलेटिव साल में 3 साल में देखोगे आपको लगेगा यूएस विल अंडर परफॉर्म। सेकंड इंडिया विल परफॉर्म बेटर देन अदर इमर्जिंग मार्केट्स। बेटर का मतलब क्या होता है? क्वार्टाइल वन में आपकी परफॉर्मेंस आ जाएगी जो अभी क्वार्टल फोर में चल रही है। इंडिया विल कम इन क्वटाइल वन परफॉर्मेंस। तीसरा मुझे लिखने में थोड़ा सा रिस्की लग रहा है और यह लास्ट है। अभी जैसे आईएआर डेप्रिशिएट हो रहा है ना हम यह है अगले एक साल के लिए वन ईयर का आई एम एक्सपेक्टिंग कि आईएआर सपोर्टेड एसेट्स विल परफॉर्म बेटर देन एनी अदर एसेट्स एंड आईएआर विल अप्रिशिएट इज अ डिफिकल्ट वर्ड टू से बट एटलीस्ट स्टेबलाइज जितना लोग बात कर रहे हैं ना $120 पे पहुंच जाएगा 130 बार भाई नहीं पहुंचेगा यूएस हैज़ परफॉर्म्ड यूएस एंड इट इज़ नथिंग कि

Output Parser

In [77]:
parser = StrOutputParser()


 Complete RAG Chain

In [78]:
main_chain = (

    parallel_chain|prompt|llm|parser

)


 Ask Question

In [79]:
question = "Can you summarize the video?"

response = main_chain.invoke(
    question
)

print()
print("FINAL RESPONSE:")
print(response)


FINAL RESPONSE:
The video discusses the concept of investing in stocks, specifically focusing on the process of listing and the potential risks involved. It covers topics such as listing strategies, the importance of fundamental analysis, and the challenges of making losses in the stock market.


In [80]:
question = """
What are the main topics discussed in this video?
"""
response = main_chain.invoke(
    question
)
print()
print("QUESTION:")
print(question)
print()
print("ANSWER:")
print(response)


QUESTION:

What are the main topics discussed in this video?


ANSWER:
The main topics discussed in this video include:

1. Investment strategy for buying and holding stocks.
2. The concept of "buy and hold" as an investment approach.
3. The importance of market timing in investing.
4. The role of different types of stocks (retail, small-cap, large-cap) in an investment portfolio.
5. The idea of "buying low and selling high" as a strategy.
6. The concept of "buying the dip" to take advantage of temporary price drops.
7. The importance of diversification in an investment portfolio.
8. The potential risks and rewards of investing in stocks.
9. The role of market data in making investment decisions.
10. The concept of "market cycles" and how they impact investment strategies.
